In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import dask
import zarr
import xarray as xr
import cftime
import os

In [2]:
# # ============== LOAD LLC ==============
# llc_path = '/orcd/data/abodner/002/cody/LLC_patch/LLC4320_face1_i2880-3600_j720-1440.zarr'
# llc_patch_full = xr.open_zarr(llc_path)

llc_patch_full = xr.open_zarr('/orcd/data/abodner/003/LLC4320/LLC4320',consolidated=False).isel(
    face=1,
    i=slice(2880,3600),
    i_g=slice(2880,3600),
    j=slice(720,1440),
    j_g=slice(720,1440),
)
# quad: i=[2880:4320), j=[0:1440)
# Agulhas: i=[2880:3600), j=[720:1440)
# ============== LOAD EMULATORS ==============
emulator_configs = [
    {
        'name': '720x720',
        'key': 'emulator_1',
        'path': '/orcd/data/abodner/002/cody/inference_patch/2026-07-20-eval:Samudra_LLC:rb-Agulhas-pred_resid-eager-ckpt50-fixed-3weeks-18427424/predictions_4d_extended-18444486.zarr',
        'desc': ''
    },
    # {
    #     'name': 'tiles-hard_stitch_ckpt7',
    #     'key': 'emulator_2',
    #     'path': '/orcd/data/abodner/002/cody/inference_patch/2026-08-12-eval:Samudra_LLC:4-tile-blend=false-20307545/predictions_4d.zarr',
    #     'desc': ''
    # },
    #     {
    #     'name': 'tiles-kbd',
    #     'key': 'emulator_3',
    #     'path': '/orcd/data/abodner/002/cody/inference_patch/2026-08-12-eval:Samudra_LLC:4-tile-blend=true,kbd,pre-blend_summary-20307546/predictions_4d.zarr',
    #     'desc': ''
    # },
    # {
    #     'name': 'tiles_quintic',
    #     'key': 'emulator_4',
    #     'path': '/orcd/data/abodner/002/cody/inference_patch/2026-08-12-eval:Samudra_LLC:4-tile-blend=true,quintic,pre-blend_summary-20307547/predictions_4d.zarr',
    #     'desc': ''
    # },
    # {
    #     'name': 'tiles_quintic-perturbations',
    #     'key': 'emulator_5',
    #     'path': '/orcd/data/abodner/002/cody/inference_patch/2026-08-09-eval:Samudra_LLC:4-tile-blend=true,quintic,far-field-perturbation-20033939/predictions_4d.zarr',
    #     'desc': ''
    # },

]

# ============== OPEN EMULATOR DATASETS ==============
emulator_patches_raw = {}
for cfg in emulator_configs:
    emulator_patches_raw[cfg['key']] = xr.open_dataset(cfg['path'], consolidated=True)
    print(f"Loaded {cfg['name']}: {cfg['desc']}")

# ============== TIME MATCHING ==============
def normalize_times(times):
    return pd.DatetimeIndex([
        pd.Timestamp(
            int(t.year), int(t.month), int(t.day),
            int(t.hour), int(t.minute), int(t.second)
        )
        if hasattr(t, "year")
        else pd.Timestamp(t).floor("s")
        for t in times
    ])

llc_times_norm = normalize_times(llc_patch_full.time.values)

common_times = llc_times_norm
for cfg in emulator_configs:
    emulator_times_norm = normalize_times(emulator_patches_raw[cfg['key']].time.values)
    common_times = common_times.intersection(emulator_times_norm)

common_times = common_times.sort_values()

llc_mask = llc_times_norm.isin(common_times)
llc_patch = llc_patch_full.isel(time=llc_mask)

print(f"LLC subset to {len(common_times)} common times")

# ============== PORT GRID VARS & BUILD UNIFIED STRUCTURE ==============
grid_vars = ['XC', 'YC', 'rA', 'Z', 'dxC', 'dyC', 'dxG', 'dyG']

emulator_patches = {}
for cfg in emulator_configs:
    patch_raw = emulator_patches_raw[cfg['key']]
    patch_times_norm = normalize_times(patch_raw.time.values)

    patch_mask = patch_times_norm.isin(common_times)
    patch = patch_raw.isel(time=patch_mask)

    for gv in grid_vars:
        patch[gv] = llc_patch[gv]

    emulator_patches[cfg['key']] = patch

# ============== UNIFIED REFERENCE LISTS ==============
emulator_info = [(cfg['name'], cfg['key']) for cfg in emulator_configs]
n_emulators = len(emulator_info)

all_patches = {'llc': llc_patch}
all_patches.update(emulator_patches)

print(f"\n=== Setup complete: LLC + {n_emulators} emulators ===")
for name, key in emulator_info:
    print(f"  {name} ({key})")

Loaded 720x720: 
LLC subset to 696 common times

=== Setup complete: LLC + 1 emulators ===
  720x720 (emulator_1)


In [3]:
selected_time_range = [0, 419]   # inclusive indices
stepping = 6                    # 1 = every timestep, 4 = every 4th timestep

start_idx, end_idx = selected_time_range

# ----------------------------------------
# First subset LLC
# ----------------------------------------
llc_patch = llc_patch.isel(
    time=slice(start_idx, end_idx + 1, stepping)
)

# ----------------------------------------
# Then subset each emulator safely
# Handles shorter emulator runs automatically
# ----------------------------------------
emulator_patches_subset = {}

for key, patch in emulator_patches.items():

    max_time = patch.sizes['time']

    # Prevent indexing past emulator length
    safe_end_idx = min(end_idx, max_time - 1)

    patch_subset = patch.isel(
        time=slice(start_idx, safe_end_idx + 1, stepping)
    )

    emulator_patches_subset[key] = patch_subset

emulator_patches = emulator_patches_subset

# ----------------------------------------
# Match LLC length to shortest emulator
# ----------------------------------------
min_time_len = min(
    [llc_patch.sizes['time']] +
    [patch.sizes['time'] for patch in emulator_patches.values()]
)

llc_patch = llc_patch.isel(time=slice(0, min_time_len))

emulator_patches = {
    key: patch.isel(time=slice(0, min_time_len))
    for key, patch in emulator_patches.items()
}

# ----------------------------------------
# Rebuild combined dict
# ----------------------------------------
all_patches = {'llc': llc_patch}
all_patches.update(emulator_patches)

# ----------------------------------------
# Diagnostics
# ----------------------------------------
print(f"Subset to time indices {start_idx}:{end_idx}")
print(f"Stepping = {stepping}")
print(f"Final synchronized length = {min_time_len}")

print(f"LLC now has {llc_patch.sizes['time']} times")

for name, key in emulator_info:
    print(
        f"{name} ({key}) now has "
        f"{emulator_patches[key].sizes['time']} times"
    )

Subset to time indices 0:419
Stepping = 6
Final synchronized length = 70
LLC now has 70 times
720x720 (emulator_1) now has 70 times
tiles_quintic (emulator_4) now has 70 times


In [4]:
def format_time(t_val):
    """Format a time value to DD/MM/YYYY:HH regardless of cftime or datetime64."""
    try:
        return f"{t_val.day:02d}/{t_val.month:02d}/{t_val.year}:{t_val.hour:02d}h"
    except AttributeError:
        t_pd = pd.Timestamp(t_val)
        return f"{t_pd.day:02d}/{t_pd.month:02d}/{t_pd.year}:{t_pd.hour:02d}h"

# surface field and field difference

In [5]:
# ============== SET VARIABLES HERE ==============
prog_vars = ['Theta']#, 'Salt', 'U', 'V']
colormaps = {'Theta': 'Spectral_r'}#, 'Salt': 'viridis', 'U': 'bwr', 'V': 'bwr'}

# prog_vars = ['Theta']
# colormaps = {'Theta': 'Spectral_r'}
# ================================================
for var in prog_vars:
    print(f"Generating plots for {var}...")
    
    os.makedirs(f'figs/prognostic_var_comparison/{var}', exist_ok=True)
    
    ref_patch = emulator_patches[emulator_info[0][1]]
    n_times = len(ref_patch.time)
    time_indices = list(range(n_times))
    nrows = len(time_indices)
    ncols_fields = 1 + n_emulators  # LLC + emulators
    ncols_diff = n_emulators         # emulators only
    
    cmap = colormaps[var]
    
    # ==================== PLOT 1: Surface fields ====================
    fig, axes = plt.subplots(nrows, ncols_fields, figsize=(3.6*ncols_fields, 3*nrows), dpi=200)
    
    if nrows == 1:
        axes = axes.reshape(1, -1)
    if ncols_fields == 1:
        axes = axes.reshape(-1, 1)
    
    for row, t in enumerate(time_indices):
        time_str = format_time(ref_patch.time.values[t])
        
        # Collect all surface fields
        fields = [llc_patch.isel(time=t, k=0)[var]]
        for emu_name, emu_key in emulator_info:
            fields.append(emulator_patches[emu_key].isel(time=t, k=0)[var])
        
        finite_fields = [f.values[np.isfinite(f.values)] for f in fields]
        if not any(vals.size for vals in finite_fields):
            print(f"  Skipping {var} {time_str}: no finite values")
            continue
        vmin = np.min([vals.min() for vals in finite_fields if vals.size])
        vmax = np.max([vals.max() for vals in finite_fields if vals.size])
        
        labels = ['LLC'] + [name for name, _ in emulator_info]
        
        for col, (field, label) in enumerate(zip(fields, labels)):
            ax = axes[row, col]
            field_values = np.ma.masked_invalid(field.values)
            cf = ax.contourf(field.coords.get('i', np.arange(field.shape[-1])),
                             field.coords.get('j', np.arange(field.shape[-2])), field_values,
                             cmap=cmap, vmin=vmin, vmax=vmax, levels=30)
            ax.set_title(f'{label} {var} {time_str}', fontsize=8)
            plt.colorbar(cf, ax=ax)



    
    
    plt.tight_layout()
    plt.savefig(f'figs/prognostic_var_comparison/{var}/surface_{var}_fields.png')
    plt.close()
    
    # # ==================== PLOT 2: Difference fields ====================
    # fig, axes = plt.subplots(nrows, ncols_diff, figsize=(4*ncols_diff, 3*nrows), dpi=200)
    
    # if nrows == 1 and ncols_diff > 1:
    #     axes = axes.reshape(1, -1)
    # elif nrows > 1 and ncols_diff == 1:
    #     axes = axes.reshape(-1, 1)
    # elif nrows == 1 and ncols_diff == 1:
    #     axes = axes.reshape(1, 1)
    
    # for row, t in enumerate(time_indices):
    #     time_str = format_time(ref_patch.time.values[t])
        
    #     llc_vis = llc_patch.isel(time=t, k=0)[var]
        
    #     diffs = []
    #     for emu_name, emu_key in emulator_info:
    #         emu_vis = emulator_patches[emu_key].isel(time=t, k=0)[var]
    #         diffs.append(llc_vis.values - emu_vis.values)
        
    #     abs_max = np.max([np.abs(d).max() for d in diffs])
    #     vmin_d, vmax_d = -abs_max, abs_max
        
    #     row_axes = [axes[row, col] for col in range(ncols_diff)]
        
    #     for col, ((emu_name, _), diff) in enumerate(zip(emulator_info, diffs)):
    #         ax = row_axes[col]
    #         cf = ax.contourf(llc_vis.coords.get('i', np.arange(llc_vis.shape[-1])),
    #                          llc_vis.coords.get('j', np.arange(llc_vis.shape[-2])), diff,
    #                          cmap="bwr", vmin=vmin_d, vmax=vmax_d, levels=30)
    #         short_name = emu_name.replace('Emulator ', 'Em')
    #         ax.set_title(f'LLC - {short_name} {var} {time_str}', fontsize=8)
        
    #     fig.colorbar(cf, ax=row_axes, orientation='vertical',
    #                  fraction=0.046, pad=0.04)
    
    # plt.savefig(f'figs/prognostic_var_comparison/{var}/surface_{var}_differences.png')
    # plt.close()
    
    print(f"✓ Saved plots for {var}")

Generating plots for Theta...
✓ Saved plots for Theta


In [12]:
prog_vars = ['Theta']
colormaps = {'Theta': 'Spectral_r'}

# ================================================

for var in prog_vars:
    print(f"Generating plots for {var}...")

    os.makedirs(
        f'figs/prognostic_var_comparison/{var}',
        exist_ok=True
    )

    ref_patch = emulator_patches[emulator_info[0][1]]

    n_times = len(ref_patch.time)
    time_indices = list(range(n_times))

    nrows = len(time_indices)
    ncols_fields = 1 + n_emulators

    cmap = colormaps[var]

    # ============================================================
    # INDEX range to display
    # ============================================================
    xmin, xmax = 328, 392

    # ============================================================
    # Helper: determine horizontal/vertical dimensions
    # ============================================================
    def get_xy_dims(field):
        """
        LLC fields may use:
            ('lat', 'lon')

        Emulator fields may use:
            ('j', 'i')

        Return:
            ydim, xdim
        """

        if 'i' in field.dims:
            xdim = 'i'
        elif 'lon' in field.dims:
            xdim = 'lon'
        else:
            xdim = field.dims[-1]

        if 'j' in field.dims:
            ydim = 'j'
        elif 'lat' in field.dims:
            ydim = 'lat'
        else:
            ydim = field.dims[-2]

        return ydim, xdim


    # ============================================================
    # PLOT: Surface fields
    # ============================================================

    fig, axes = plt.subplots(
        nrows,
        ncols_fields,

        # Wide + short = strip appearance
        figsize=(1.0 * ncols_fields, 15 * nrows),

        dpi=200,
        squeeze=False
    )


    for row, t in enumerate(time_indices):

        time_str = format_time(ref_patch.time.values[t])

        # --------------------------------------------------------
        # Gather LLC + emulator fields
        # --------------------------------------------------------
        fields = [
            llc_patch.isel(time=t, k=0)[var]
        ]

        for emu_name, emu_key in emulator_info:
            fields.append(
                emulator_patches[emu_key]
                .isel(time=t, k=0)[var]
            )

        labels = (
            ['LLC']
            + [name for name, _ in emulator_info]
        )

        # ========================================================
        # Find color limits using ONLY indices 328:392
        # ========================================================

        zoomed_values = []

        for field in fields:

            ydim, xdim = get_xy_dims(field)

            # IMPORTANT:
            # Slice by INDEX, not coordinate value.
            #
            # Python excludes the upper endpoint, hence xmax + 1.
            field_zoom = field.isel(
                {xdim: slice(xmin, xmax + 1)}
            )

            vals = np.asarray(field_zoom.values)

            vals = vals[np.isfinite(vals)]

            if vals.size > 0:
                zoomed_values.append(vals)


        if not zoomed_values:
            print(
                f"  Skipping {var} {time_str}: "
                "no finite values"
            )
            continue


        # Shared scale across LLC + emulators
        # for THIS time step
        vmin = min(
            vals.min()
            for vals in zoomed_values
        )

        vmax = max(
            vals.max()
            for vals in zoomed_values
        )

        levels = np.linspace(
            vmin,
            vmax,
            31
        )


        # ========================================================
        # Plot each field
        # ========================================================

        for col, (field, label) in enumerate(
            zip(fields, labels)
        ):

            ax = axes[row, col]

            ydim, xdim = get_xy_dims(field)

            # ----------------------------------------------------
            # Slice horizontal dimension by INDEX
            # ----------------------------------------------------

            field_zoom = field.isel(
                {xdim: slice(xmin, xmax + 1)}
            )

            field_values = np.ma.masked_invalid(
                np.asarray(field_zoom.values)
            )


            # ----------------------------------------------------
            # Use simple index coordinates for plotting.
            #
            # This avoids mixing LLC lon/lat coordinates with
            # emulator i/j coordinates.
            # ----------------------------------------------------

            x_plot = np.arange(
                xmin,
                xmin + field_zoom.sizes[xdim]
            )

            y_plot = np.arange(
                field_zoom.sizes[ydim]
            )


            # ----------------------------------------------------
            # Plot
            # ----------------------------------------------------

            cf = ax.contourf(
                x_plot,
                y_plot,
                field_values,

                levels=levels,
                cmap=cmap,
                extend='both'
            )


            # ----------------------------------------------------
            # Force the requested zoom
            # ----------------------------------------------------

            ax.set_xlim(
                xmin,
                xmax
            )


            # ----------------------------------------------------
            # Strip geometry
            #
            # "auto" is intentional here. The physical subplot
            # dimensions determine the strip appearance instead
            # of forcing equal x/y units.
            # ----------------------------------------------------

            ax.set_aspect('auto')


            # ----------------------------------------------------
            # Titles
            # ----------------------------------------------------

            ax.set_title(
                f'{label} {var} | {time_str}',
                fontsize=8,
                pad=3
            )


            # ----------------------------------------------------
            # Axis formatting
            # ----------------------------------------------------

            if col == 0:
                ax.set_ylabel(
                    'cross-patch index',
                    fontsize=7
                )
            else:
                ax.set_ylabel('')

                # Remove repeated y tick labels
                ax.tick_params(
                    axis='y',
                    labelleft=False
                )


            if row == nrows - 1:

                ax.set_xlabel(
                    'i / horizontal index',
                    fontsize=7
                )

            else:

                ax.set_xlabel('')

                ax.tick_params(
                    axis='x',
                    labelbottom=False
                )


            ax.tick_params(
                axis='both',
                labelsize=6
            )


            # ----------------------------------------------------
            # SEPARATE colorbar for every strip
            # ----------------------------------------------------

            cbar = fig.colorbar(
                cf,
                ax=ax,
                orientation='vertical',

                # Narrow colorbar
                fraction=0.025,

                # Small gap between strip and colorbar
                pad=0.015,

                aspect=25
            )

            cbar.ax.tick_params(
                labelsize=5
            )


    # ============================================================
    # Spacing
    # ============================================================

    fig.subplots_adjust(
        left=0.06,
        right=0.98,
        bottom=0.06,
        top=0.96,

        # Space for individual colorbars
        wspace=0.28,

        # Small gap between strips
        hspace=0.40
    )


    # ============================================================
    # Save
    # ============================================================

    plt.savefig(
        f'figs/prognostic_var_comparison/{var}/'
        f'surface_{var}_fields_zoomed.png',

        bbox_inches='tight'
    )

    plt.close()

    print(f"✓ Saved plots for {var}")

Generating plots for Theta...
✓ Saved plots for Theta


# Gradients

In [8]:
grad_vars = ['Theta', 'Salt', 'U', 'V']

metric_vars = {
    'Theta': (None, None),
    'Salt': (None, None),
    'U': ('dxC', 'dyG'),   # U is on (j, i_g)
    'V': ('dxG', 'dyC'),   # V is on (j_g, i)
}


def _metric_values(patch, var):
    dx_name, dy_name = metric_vars[var]
    if dx_name is None or dy_name is None:
        dx = np.sqrt(patch['rA'].values)  # tracer variables are on (j, i)
        dy = dx
    else:
        dx = patch[dx_name].values
        dy = patch[dy_name].values
    return dx, dy


for var in grad_vars:
    grad_name = f'grad_{var}'
    print(f"Computing {grad_name}...")
    
    for patch_name, patch in all_patches.items():
        da_var = patch[var]
        data = da_var.values  # expected: (time, k, y, x)
        dx, dy = _metric_values(patch, var)
        
        if dx.shape != data.shape[-2:] or dy.shape != data.shape[-2:]:
            raise ValueError(
                f"{patch_name} {var}: data horizontal shape {data.shape[-2:]} "
                f"does not match dx {dx.shape} / dy {dy.shape}"
            )
        
        d_di = (np.roll(data, -1, axis=-1) - np.roll(data, 1, axis=-1)) / (2 * dx[np.newaxis, np.newaxis, :, :])
        d_dj = (np.roll(data, -1, axis=-2) - np.roll(data, 1, axis=-2)) / (2 * dy[np.newaxis, np.newaxis, :, :])
        
        grad_mag = np.sqrt(d_di**2 + d_dj**2)
        
        patch[grad_name] = (da_var.dims, grad_mag)
        print(f"  ✓ {patch_name} {grad_name}: {grad_mag.shape} on {da_var.dims[-2:]}")

print("Done computing gradients!")

Computing grad_Theta...
  ✓ llc grad_Theta: (5, 51, 720, 720) on ('j', 'i')
  ✓ emulator_1 grad_Theta: (5, 51, 720, 720) on ('lat', 'lon')
Computing grad_Salt...
  ✓ llc grad_Salt: (5, 51, 720, 720) on ('j', 'i')
  ✓ emulator_1 grad_Salt: (5, 51, 720, 720) on ('lat', 'lon')
Computing grad_U...
  ✓ llc grad_U: (5, 51, 720, 720) on ('j', 'i_g')
  ✓ emulator_1 grad_U: (5, 51, 720, 720) on ('lat', 'lon')
Computing grad_V...
  ✓ llc grad_V: (5, 51, 720, 720) on ('j_g', 'i')
  ✓ emulator_1 grad_V: (5, 51, 720, 720) on ('lat', 'lon')
Done computing gradients!


In [9]:
gradient_masks = {}

for patch_name, patch in all_patches.items():
    gradient_masks[patch_name] = {}
    
    for var in grad_vars:
        grad_name = f'grad_{var}'
        grad_data = patch[grad_name].values  # (time, k, j, i)
        
        n_times, n_depths = grad_data.shape[0], grad_data.shape[1]
        mask = np.zeros_like(grad_data, dtype=bool)
        
        for t in range(n_times):
            for k in range(n_depths):
                field = grad_data[t, k]
                threshold = np.nanpercentile(field, 99)
                mask[t, k] = field >= threshold
        
        gradient_masks[patch_name][var] = mask
        print(f"✓ {patch_name} {var}: {mask.sum()} high-gradient pixels ({mask.sum() / mask.size * 100:.1f}%)")

print("Done creating gradient masks!")

✓ llc Theta: 1322065 high-gradient pixels (1.0%)
✓ llc Salt: 1322065 high-gradient pixels (1.0%)
✓ llc U: 1322045 high-gradient pixels (1.0%)
✓ llc V: 1322045 high-gradient pixels (1.0%)
✓ emulator_1 Theta: 1322176 high-gradient pixels (1.0%)
✓ emulator_1 Salt: 1322175 high-gradient pixels (1.0%)
✓ emulator_1 U: 1322175 high-gradient pixels (1.0%)
✓ emulator_1 V: 1322175 high-gradient pixels (1.0%)
Done creating gradient masks!


In [10]:
drift_colours = {
    'llc_only': '#FC9065',   # orange
    'emu_only': '#BE3977',   # pink
    'overlap':  '#FCFCBE',   # light tan
}

In [11]:
for var in grad_vars:
    print(f"Generating gradient drift figure for {var}...")
    
    os.makedirs(f'figs/gradients/{var}', exist_ok=True)
    
    ref_patch = emulator_patches[emulator_info[0][1]]
    n_times = len(ref_patch.time)
    time_indices = list(range(n_times))
    nrows = len(time_indices)
    ncols = n_emulators
    
    fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 3.5*nrows), dpi=150)
    
    if nrows == 1 and ncols > 1:
        axes = axes.reshape(1, -1)
    elif nrows > 1 and ncols == 1:
        axes = axes.reshape(-1, 1)
    elif nrows == 1 and ncols == 1:
        axes = axes.reshape(1, 1)
    
    # Get coordinate blocks for lat/lon ticks
    def _coord_block(patch, name, n_j, n_i):
        coord = patch[name]
        j_dim = 'j' if 'j' in coord.dims else ('lat' if 'lat' in coord.dims else 'y')
        i_dim = 'i' if 'i' in coord.dims else ('lon' if 'lon' in coord.dims else 'x')
        sel = coord.isel({
            j_dim: slice(0, n_j),
            i_dim: slice(0, n_i),
        })
        for extra_dim in set(sel.dims) - {j_dim, i_dim}:
            sel = sel.isel({extra_dim: 0})
        return sel.transpose(j_dim, i_dim).values
    
    def _nice_lon_lat_ticks(coord_2d, axis_vals, axis):
        idx = np.linspace(0, len(axis_vals) - 1, 5).round().astype(int)
        pos = axis_vals[idx]
        if axis == 'i':
            coord_vals = coord_2d[coord_2d.shape[0] // 2, idx]
        else:
            coord_vals = coord_2d[idx, coord_2d.shape[1] // 2]
        return pos, [f'{v:.1f}' for v in coord_vals]
    
    for row, t in enumerate(time_indices):
        time_str = format_time(ref_patch.time.values[t])
        
        llc_mask_surface = gradient_masks['llc'][var][t, 0]  # (j, i)
        
        # Get coordinate blocks and ticks
        n_j, n_i = llc_mask_surface.shape
        j_vals = np.arange(n_j)
        i_vals = np.arange(n_i)
        
        xc_block = _coord_block(llc_patch, 'XC', n_j, n_i)
        yc_block = _coord_block(llc_patch, 'YC', n_j, n_i)
        x_tick_pos, x_tick_labels = _nice_lon_lat_ticks(xc_block, i_vals, 'i')
        y_tick_pos, y_tick_labels = _nice_lon_lat_ticks(yc_block, j_vals, 'j')
        
        for col, (emu_name, emu_key) in enumerate(emulator_info):
            ax = axes[row, col]
            
            emu_field = all_patches[emu_key].isel(time=t, k=0)[var].values
            emu_mask_surface = gradient_masks[emu_key][var][t, 0]
            
            ax.imshow(emu_field, cmap='Greys', aspect='auto', origin='lower')
            
            overlap_mask = llc_mask_surface & emu_mask_surface
            llc_only_mask = llc_mask_surface & ~emu_mask_surface
            emu_only_mask = emu_mask_surface & ~llc_mask_surface
            
            llc_j, llc_i = np.where(llc_only_mask)
            emu_j, emu_i = np.where(emu_only_mask)
            ovl_j, ovl_i = np.where(overlap_mask)
            
            ax.scatter(llc_i, llc_j, c=drift_colours['llc_only'], s=1, alpha=0.5, label='LLC top 2.5%', rasterized=True)
            ax.scatter(emu_i, emu_j, c=drift_colours['emu_only'], s=1, alpha=0.5, label='Emu top 2.5%', rasterized=True)
            ax.scatter(ovl_i, ovl_j, c=drift_colours['overlap'], s=1, alpha=0.7, label='Overlap', rasterized=True)
            
            n_overlap = np.sum(overlap_mask)
            pix = int(0.025 * np.size(all_patches['emulator_1'].isel(time=t, k=0)[var].lat) * np.size(all_patches['emulator_1'].isel(time=t, k=0)[var].lon))
            ax.set_title(f'{emu_name} {var} {time_str} overlap={n_overlap}/{pix}', fontsize=8)
            
            ax.set_xticks(x_tick_pos)
            ax.set_xticklabels(x_tick_labels, fontsize=7)
            ax.set_yticks(y_tick_pos)
            ax.set_yticklabels(y_tick_labels, fontsize=7)
            ax.set_xlabel('Longitude', fontsize=7)
            ax.set_ylabel('Latitude', fontsize=7)
            
            # if col == ncols - 1:
            #     ax.legend(fontsize=5, loc='upper right', markerscale=5)
        
    
    plt.tight_layout()
    plt.savefig(f'figs/gradients/{var}/surface_gradient_drift.png', dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"✓ Saved gradient drift figure for {var}")

print("Done with gradient drift figures!")

Generating gradient drift figure for Theta...
✓ Saved gradient drift figure for Theta
Generating gradient drift figure for Salt...
✓ Saved gradient drift figure for Salt
Generating gradient drift figure for U...
✓ Saved gradient drift figure for U
Generating gradient drift figure for V...
✓ Saved gradient drift figure for V
Done with gradient drift figures!


# Error vs depth plots

In [ ]:
error_colours = {
    'mean': '#631980',  # dark purple
    'median': '#BE3977',   # pink
    'high-gradient-mean':  '#FC9065',   # orange
    'std': 'k' # 
}
depth_vars = ['Theta', 'Salt', 'U', 'V']
ref_lines = {
    # 'Theta': [0.5, 1.0],
    # 'Salt': [0.06, 0.12],
    # 'U': [0.075, 0.15],
    # 'V': [0.075, 0.15],
}

for var in depth_vars:
    print(f"Generating augmented depth error plots for {var}...")
    
    os.makedirs(f'figs/prognostic_var_comparison/{var}', exist_ok=True)
    
    n_depths = llc_patch.sizes['k']
    ref_patch = emulator_patches[emulator_info[0][1]]
    n_times = len(ref_patch.time)
    time_indices = list(range(n_times))
    
    nrows = len(time_indices)
    ncols = n_emulators
    
    fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 3*nrows), dpi=150)
    
    if nrows == 1 and ncols > 1:
        axes = axes.reshape(1, -1)
    elif nrows > 1 and ncols == 1:
        axes = axes.reshape(-1, 1)
    elif nrows == 1 and ncols == 1:
        axes = axes.reshape(1, 1)
    
    depths = -1 * np.round(llc_patch.Z.values).astype(int)
    
    for row, t in enumerate(time_indices):
        time_str = format_time(ref_patch.time.values[t])
        
        llc_data = llc_patch.isel(time=t)[var].values  # (k, j, i)
        llc_grad_mask = gradient_masks['llc'][var][t]   # (k, j, i)
        
        # First pass: compute all errors for shared xlim
        row_mean_errors = []
        row_median_errors = []
        row_hg_mean_errors = []
        row_std_errors = []
        
        for emu_name, emu_key in emulator_info:
            emu_data = emulator_patches[emu_key].isel(time=t)[var].values
            diff = np.abs(llc_data - emu_data)
            
            diff_flat = diff.reshape(n_depths, -1)
            mean_errors = np.nanmean(diff_flat, axis=1)
            median_errors = np.nanmedian(diff_flat, axis=1)
            std_errors = np.nanstd(diff_flat, axis=1)
            hg_mean_errors = np.zeros(n_depths)
            for k in range(n_depths):
                hg_pixels = diff[k][llc_grad_mask[k]]
                if len(hg_pixels) > 0:
                    hg_mean_errors[k] = np.nanmean(hg_pixels)
                else:
                    hg_mean_errors[k] = np.nan
            
            row_mean_errors.append(mean_errors)
            row_median_errors.append(median_errors)
            row_hg_mean_errors.append(hg_mean_errors)
            row_std_errors.append(std_errors)
        all_errors = np.concatenate(
            row_mean_errors
            + row_median_errors
            + row_hg_mean_errors
            + [mean + std for mean, std in zip(row_mean_errors, row_std_errors)]
        )
        xmin = 0
        xmax = np.nanmax(all_errors) * 1.05
        
        # Second pass: plot
        for col, (emu_name, _) in enumerate(emulator_info):
            ax = axes[row, col]
            
            ax.scatter(row_median_errors[col], depths, color=f'{error_colours["median"]}', s=30, alpha=0.7, zorder=3)
            ax.plot(row_median_errors[col], depths, color=f'{error_colours["median"]}', alpha=0.4, linewidth=1.5, label='Median')

            ax.scatter(row_mean_errors[col], depths, color=f'{error_colours["mean"]}', s=30, alpha=0.7, zorder=4)
            ax.plot(row_mean_errors[col], depths, color=f'{error_colours["mean"]}', alpha=0.4, linewidth=1.5, label='Mean')
            
            ax.scatter(row_hg_mean_errors[col], depths, color=f'{error_colours["high-gradient-mean"]}', s=30, alpha=0.7, zorder=5)
            ax.plot(row_hg_mean_errors[col], depths, color=f'{error_colours["high-gradient-mean"]}', alpha=0.4, linewidth=1.5, label='HG Mean')

            ax.errorbar(
                row_mean_errors[col],
                depths,
                xerr=row_std_errors[col],
                fmt='none',
                ecolor=error_colours['std'],
                elinewidth=1.0,
                capsize=2,
                alpha=0.8,
                label='Mean +/- Std',
                zorder=6,
            )

            
            # for ref_val in ref_lines[var]:
            #     ax.axvline(x=ref_val, color='black', linestyle='--', linewidth=1.5, alpha=0.5, zorder=2)
            
            ax.set_title(f'{emu_name} {var} {time_str}', fontsize=8)
            ax.set_xlabel('Abs Error', fontsize=7)
            ax.set_ylabel('Depth (m)', fontsize=7)
            ax.set_ylim(1000, 0)
            ax.set_yticks([0, 250, 500, 750, 1000])
            ax.set_xlim(xmin, xmax)
            ax.grid(alpha=0.2)
            ax.tick_params(labelsize=6)
            
            if col == 0:
                ax.legend(fontsize=6, loc='lower right')

            ax.set_facecolor('#E3E3E3')
    
    plt.tight_layout()
    plt.savefig(f'figs/prognostic_var_comparison/{var}/depth_error_by_time.png', dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"✓ Saved augmented depth error plots for {var}")

print("Done!")


Generating augmented depth error plots for Theta...
✓ Saved augmented depth error plots for Theta
Generating augmented depth error plots for Salt...
✓ Saved augmented depth error plots for Salt
Generating augmented depth error plots for U...
✓ Saved augmented depth error plots for U
Generating augmented depth error plots for V...


In [ ]:
error_colours = {
    'mean': '#631980',  # dark purple
    'median': '#BE3977',   # pink
    'high-gradient-mean':  '#FC9065',   # orange
    'std': 'k' # black error bars
}
depth_vars = ['Theta']#, 'Salt', 'U', 'V']
ref_lines = {
    'Theta': [0.5, 1.0],
    # 'Salt': [0.06, 0.12],
    # 'U': [0.075, 0.15],
    # 'V': [0.075, 0.15],
}

for var in depth_vars:
    print(f"Generating augmented depth error plots for {var}...")
    
    os.makedirs(f'figs/prognostic_var_comparison/{var}', exist_ok=True)
    
    n_depths = llc_patch.sizes['k']
    ref_patch = emulator_patches[emulator_info[0][1]]
    n_times = len(ref_patch.time)
    time_indices = list(range(n_times))
    
    nrows = len(time_indices)
    ncols = n_emulators
    
    fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 3*nrows), dpi=150)
    
    if nrows == 1 and ncols > 1:
        axes = axes.reshape(1, -1)
    elif nrows > 1 and ncols == 1:
        axes = axes.reshape(-1, 1)
    elif nrows == 1 and ncols == 1:
        axes = axes.reshape(1, 1)
    
    depths_full = np.round(llc_patch.Z.values).astype(int)
    k_indices = np.where((depths_full <= 0) & (depths_full >= -100))[0]
    depths = depths_full[k_indices]
    
    for row, t in enumerate(time_indices):
        time_str = format_time(ref_patch.time.values[t])
        
        llc_data = llc_patch.isel(time=t, k=k_indices)[var].values  # (k, j, i)
        llc_grad_mask = gradient_masks['llc'][var][t]   # (k, j, i)
        llc_grad_mask = llc_grad_mask[k_indices]
        
        # First pass: compute all errors for shared xlim
        row_mean_errors = []
        row_median_errors = []
        row_hg_mean_errors = []
        row_std_errors = []
        
        for emu_name, emu_key in emulator_info:
            emu_data = emulator_patches[emu_key].isel(time=t, k=k_indices)[var].values
            diff = np.abs(llc_data - emu_data)
            
            n_depths_subset = len(k_indices)
            diff_flat = diff.reshape(n_depths_subset, -1)
            mean_errors = np.nanmean(diff_flat, axis=1)
            median_errors = np.nanmedian(diff_flat, axis=1)
            std_errors = np.nanstd(diff_flat, axis=1)
            hg_mean_errors = np.zeros(n_depths_subset)
            for k in range(n_depths_subset):
                hg_pixels = diff[k][llc_grad_mask[k]]
                if len(hg_pixels) > 0:
                    hg_mean_errors[k] = np.nanmean(hg_pixels)
                else:
                    hg_mean_errors[k] = np.nan
            
            row_mean_errors.append(mean_errors)
            row_median_errors.append(median_errors)
            row_hg_mean_errors.append(hg_mean_errors)
            row_std_errors.append(std_errors)
        all_errors = np.concatenate(
            row_mean_errors
            + row_median_errors
            + row_hg_mean_errors
            + [mean + std for mean, std in zip(row_mean_errors, row_std_errors)]
        )
        xmin = 0
        xmax = np.nanmax(all_errors) * 1.05
        
        # Second pass: plot
        for col, (emu_name, _) in enumerate(emulator_info):
            ax = axes[row, col]
            
            ax.scatter(row_median_errors[col], depths, color=f'{error_colours["median"]}', s=30, alpha=0.7, zorder=3)
            ax.plot(row_median_errors[col], depths, color=f'{error_colours["median"]}', alpha=0.4, linewidth=1.5, label='Median')

            ax.scatter(row_mean_errors[col], depths, color=f'{error_colours["mean"]}', s=30, alpha=0.7, zorder=4)
            ax.plot(row_mean_errors[col], depths, color=f'{error_colours["mean"]}', alpha=0.4, linewidth=1.5, label='Mean')
            
            ax.scatter(row_hg_mean_errors[col], depths, color=f'{error_colours["high-gradient-mean"]}', s=30, alpha=0.7, zorder=5)
            ax.plot(row_hg_mean_errors[col], depths, color=f'{error_colours["high-gradient-mean"]}', alpha=0.4, linewidth=1.5, label='HG Mean')

            ax.errorbar(
                row_mean_errors[col],
                depths,
                xerr=row_std_errors[col],
                fmt='none',
                ecolor=error_colours['std'],
                elinewidth=1.0,
                capsize=2,
                alpha=0.8,
                label='Mean +/- Std',
                zorder=6,
            )

            
            # for ref_val in ref_lines[var]:
            #     ax.axvline(x=ref_val, color='black', linestyle='--', linewidth=1.5, alpha=0.5, zorder=2)
            
            ax.set_title(f'{emu_name} {var} {time_str}', fontsize=8)
            ax.set_xlabel('Abs Error', fontsize=7)
            ax.set_ylabel('Depth (m)', fontsize=7)
            ax.set_ylim(-100, 0)
            ax.set_yticks([0, -25, -50, -75, -100])
            ax.set_xlim(xmin, xmax)
            ax.grid(alpha=0.2)
            ax.tick_params(labelsize=6)
            
            if col == 0:
                ax.legend(fontsize=6, loc='lower right')

            ax.set_facecolor('#E3E3E3')
    
    plt.tight_layout()
    plt.savefig(f'figs/prognostic_var_comparison/{var}/depth_error_by_time_surface.png', dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"✓ Saved surface depth error plots for {var}")

print("Done!")
